In [ ]:
{
 "cells": [
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "# SFT v3: yangi data, Qwen2.5-1.5B, LoRA va QLoRA (Kaggle GPU)\n",
    "\n",
    "v2 bilan bir xil model va sozlamalar, faqat **data yangi** (`sft_v3/data/`): v2 -> v3 farqi datadan keladi.\n",
    "Yagona boshqa o'zgarish: 5 emas, 3 epoch (v2 da ikkala usul ham 2-epochdan keyin overfit bo'lgan).\n",
    "\n",
    "Boshlashdan oldin o'ng tomondagi **Settings** bo'limida:\n",
    "\n",
    "- **Accelerator -> GPU T4 x2** (yoki GPU P100)\n",
    "- **Internet -> On**\n",
    "\n",
    "Keyin **Run All** ni bosing."
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## 1. GPU borligini tekshirish"
   ]
  },
  {
   "cell_type": "code",
   "metadata": {},
   "source": [
    "!nvidia-smi --query-gpu=name,memory.total --format=csv"
   ],
   "execution_count": null,
   "outputs": []
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## 2. Loyiha va base modelni yuklab olish"
   ]
  },
  {
   "cell_type": "code",
   "metadata": {},
   "source": [
    "%cd /kaggle/working\n",
    "!rm -rf Post_Training\n",
    "!GIT_LFS_SKIP_SMUDGE=1 git clone -q https://github.com/davronbek-malikov/Post_Training.git\n",
    "%cd Post_Training\n",
    "!pip -q install -U transformers peft bitsandbytes accelerate\n",
    "!pip -q uninstall -y torchao  # Kaggle'dagi eski torchao peft bilan adapter yuklashni buzadi\n",
    "\n",
    "from huggingface_hub import snapshot_download\n",
    "snapshot_download(\"Qwen/Qwen2.5-1.5B\", local_dir=\"Qwen2.5-1.5B\")\n",
    "!ls -lh Qwen2.5-1.5B/*.safetensors\n",
    "!wc -l sft_v3/data/train.jsonl sft_v3/data/val.jsonl"
   ],
   "execution_count": null,
   "outputs": []
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## 3. LoRA bilan o'qitish (v3 data)"
   ]
  },
  {
   "cell_type": "code",
   "metadata": {},
   "source": [
    "!python sft_v2/train.py --method lora --data sft_v3/data --name sft-lora-v3 --epochs 3 --rank 64"
   ],
   "execution_count": null,
   "outputs": []
  },
  {
   "cell_type": "code",
   "metadata": {},
   "source": [
    "!rm -f /kaggle/working/sft_v3_adapters.zip\n",
    "!zip -q -r /kaggle/working/sft_v3_adapters.zip chat_ui/adapters/sft-lora-v3\n",
    "!ls -lh /kaggle/working/sft_v3_adapters.zip"
   ],
   "execution_count": null,
   "outputs": []
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## 4. QLoRA bilan o'qitish (v3 data)\n",
    "\n",
    "Chiqishning boshida `QLoRA: ... ta qatlam 4-bit` degan qator bo'lishi kerak."
   ]
  },
  {
   "cell_type": "code",
   "metadata": {},
   "source": [
    "!python sft_v2/train.py --method qlora --data sft_v3/data --name sft-qlora-v3 --epochs 3 --rank 64"
   ],
   "execution_count": null,
   "outputs": []
  },
  {
   "cell_type": "code",
   "metadata": {},
   "source": [
    "!zip -q -r /kaggle/working/sft_v3_adapters.zip chat_ui/adapters/sft-qlora-v3\n",
    "!ls -lh /kaggle/working/sft_v3_adapters.zip"
   ],
   "execution_count": null,
   "outputs": []
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## 5. Baholash: 68 ta test savoli\n",
    "\n",
    "v2 (1.5B, eski data): LoRA 44/68, QLoRA 42/68."
   ]
  },
  {
   "cell_type": "code",
   "metadata": {},
   "source": [
    "!python eval/run_eval.py --base Qwen2.5-1.5B --model sft-lora-v3 | tail -12\n",
    "!python eval/run_eval.py --base Qwen2.5-1.5B --model sft-qlora-v3 | tail -12\n",
    "!zip -q /kaggle/working/sft_v3_adapters.zip eval/results/sft-lora-v3.json eval/results/sft-qlora-v3.json"
   ],
   "execution_count": null,
   "outputs": []
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## 6. Yuklab olish\n",
    "\n",
    "`sft_v3_adapters.zip` ni yuklab oling va loyiha papkasida oching (yoki `sft_v3/` ichiga qo'ying)."
   ]
  },
  {
   "cell_type": "code",
   "metadata": {},
   "source": [
    "%cd /kaggle/working\n",
    "from IPython.display import FileLink\n",
    "FileLink(\"sft_v3_adapters.zip\")"
   ],
   "execution_count": null,
   "outputs": []
  }
 ],
 "metadata": {
  "kernelspec": {
   "display_name": "Python 3",
   "language": "python",
   "name": "python3"
  },
  "language_info": {
   "name": "python"
  },
  "accelerator": "GPU"
 },
 "nbformat": 4,
 "nbformat_minor": 5
}